# 09 — SHAP (validação de 2024)
Interpretação por amostra fixa de 5.000 registros, sem consulta ao teste final de 2025. O gráfico de dispersão mostra a direção e a magnitude do efeito sobre a saída do modelo (não causalidade).

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import joblib
ROOT=Path.cwd(); OUT=ROOT/"outputs"; DATA=ROOT/"data"/"processed"
SILVER=DATA/"prf_acidentes_silver_2020_2025.parquet"
for p in ["modelos","interpretabilidade","modelo_final"]:
    (OUT/p).mkdir(parents=True,exist_ok=True)
df=pd.read_parquet(SILVER)
df["br_categoria"]=pd.to_numeric(df.br,errors="coerce").astype("Int64").astype("string").fillna("Não informado")
flags=sorted(c for c in df if c.startswith("tracado_") and c!="tracado_via")
categoricas=["uf","br_categoria","dia_semana","fase_dia","sentido_via","condicao_metereologica","tipo_pista","uso_solo","tipo_acidente"]
numericas=["km","mes","hora","fim_semana","qtd_caracteristicas_tracado","veiculos"]+flags
features=categoricas+numericas
proibidas={"pessoas","mortos","feridos_graves","feridos_leves","feridos","ilesos","ignorados","classificacao_acidente","acidente_fatal"}
assert set(features).isdisjoint(proibidas)
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
def pre(cat=categoricas,num=numericas):
    try: enc=OneHotEncoder(handle_unknown="ignore",min_frequency=100,sparse_output=True)
    except TypeError: enc=OneHotEncoder(handle_unknown="ignore",min_frequency=100,sparse=True)
    return ColumnTransformer([("categoricas",enc,cat),("numericas",StandardScaler(),num)])
import shap,matplotlib.pyplot as plt
clf=joblib.load(OUT/"modelos"/"xgboost.joblib")
valid=df[df.ano==2024].sample(n=min(5000,(df.ano==2024).sum()),random_state=42)
tr=clf.named_steps["preprocessamento"]
X=tr.transform(valid[features])
X=X.toarray() if hasattr(X,"toarray") else X
nomes=tr.get_feature_names_out()
explainer=shap.TreeExplainer(clf.named_steps["modelo"])
valores=explainer.shap_values(X)
if isinstance(valores,list):valores=valores[-1]
imp=pd.DataFrame({"feature":nomes,"mean_abs_shap":np.abs(valores).mean(axis=0)}).sort_values("mean_abs_shap",ascending=False)
display(imp.head(20))
imp.to_csv(OUT/"interpretabilidade"/"shap_importance.csv",index=False)
shap.summary_plot(valores,X,feature_names=nomes,max_display=25,show=False)
plt.tight_layout();plt.savefig(OUT/"interpretabilidade"/"shap_summary_plot.png",dpi=180,bbox_inches="tight")
plt.show()